# Module 07: TensorFlow & Keras for Deep Learning
## Notebook 02: Keras Model Architectures: Sequential, Functional & Subclassing

Keras is the high-level deep learning API of TensorFlow. Keras provides an approachable, highly productive interface for constructing neural architectures while maintaining complete flexibility through three distinct authoring paradigms.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Master the trade-offs between the **Sequential API**, **Functional API**, and **Model Subclassing**.
2. Configure fundamental layers: `Dense`, `Dropout`, `BatchNormalization`, and activation functions (`relu`, `gelu`, `swish`, `softmax`).
3. Compile models with modern optimizers (`AdamW`, `Adam`, `SGD`), loss functions, and evaluation metrics.
4. **Advanced:** Construct a **Multi-Branch Deep Residual Network (ResNet-style)** with skip connections and multi-task heads via the Functional API.
5. **Advanced:** Build a fully compliant **Custom Keras Layer** from scratch with dynamic weight initialization (`build`) and tensor transformation (`call`).

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import keras
from keras import layers, models, optimizers, losses, metrics
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

TensorFlow Version: 2.21.0
Keras Version: 3.15.1


### 1. Paradigm 1: The Sequential API
The **Sequential API (`keras.Sequential`)** is designed for a single-input, single-output linear stack of layers:
- Extremely concise, clean, and readable.
- **Limitations:** Cannot express architectures with shared layers, residual / skip connections, or multiple inputs/outputs.

In [4]:
# Constructing a 3-layer MLP via Sequential API
model_seq = models.Sequential([
    layers.Input(shape=(32,)), # Explicit input shape definition
    layers.Dense(64, activation="relu", name="hidden_layer_1"),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.Dense(32, activation="relu", name="hidden_layer_2"),
    layers.Dense(4, activation="softmax", name="output_probabilities")
], name="Sequential_Classifier")

model_seq.summary()

Model: "Sequential_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_probabilities (Dense)    │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,580 (17.89 KB)

 Trainable params: 4,452 (17.39 KB)

 Non-trainable params: 128 (512.00 B)

### 2. Paradigm 2: The Functional API
The **Functional API** treats layers as callable functions that accept and return tensors:
- Expresses arbitrary Directed Acyclic Graph (DAG) topologies.
- Enables **Residual Skip Connections** ($y = F(x) + x$), multi-modal inputs, and multi-task outputs.
- Supports layer sharing and intermediate feature extraction.

In [5]:
# Multi-Input, Multi-Output Network via Functional API
# Scenario: Medical diagnostic network taking Tabular Patient Labs (16 features) + Vitals (8 features)
input_labs = layers.Input(shape=(16,), name="patient_labs")
input_vitals = layers.Input(shape=(8,), name="patient_vitals")

# Branch 1: Labs feature extractor
x1 = layers.Dense(32, activation="swish")(input_labs)
x1 = layers.BatchNormalization()(x1)

# Branch 2: Vitals feature extractor
x2 = layers.Dense(16, activation="swish")(input_vitals)
x2 = layers.BatchNormalization()(x2)

# Merge branches via concatenation
merged = layers.concatenate([x1, x2], name="multimodal_fusion")

# Shared representation
shared = layers.Dense(32, activation="swish")(merged)
shared = layers.Dropout(0.2)(shared)

# Multi-Task Heads:
# Head A: Binary risk classification (e.g. readmission risk)
out_classification = layers.Dense(1, activation="sigmoid", name="readmission_risk")(shared)

# Head B: Continuous metric regression (e.g. expected length of stay in days)
out_regression = layers.Dense(1, activation="linear", name="length_of_stay")(shared)

model_functional = models.Model(
    inputs=[input_labs, input_vitals],
    outputs=[out_classification, out_regression],
    name="MultiModal_MultiTask_Network"
)

model_functional.summary()

Model: "MultiModal_MultiTask_Network"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ patient_labs        │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patient_vitals      │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        544 │ patient_labs[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        144 │ patient_vitals[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multimodal_fusion   │ (None, 48)        │          0 │ batch_normalizat… │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      1,568 │ multimodal_fusio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ readmission_risk    │ (None, 1)         │         33 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ length_of_stay      │ (None, 1)         │         33 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,514 (9.82 KB)

 Trainable params: 2,418 (9.45 KB)

 Non-trainable params: 96 (384.00 B)

### 3. Paradigm 3: Model Subclassing
**Model Subclassing** provides maximum imperative freedom:
- Inherit from `keras.Model`.
- Define layers in `__init__()`.
- Implement forward-pass computation and dynamic control flow in `call(inputs, training=False)`.
- Ideal for research experiments involving dynamic loops, tree-structured architectures, or stateful recurrence.

In [6]:
class DynamicResidualMLP(keras.Model):
    # Fully customizable dynamic model subclass.
    def __init__(self, hidden_dim=64, num_classes=10, **kwargs):
        super().__init__(**kwargs)
        self.dense_in = layers.Dense(hidden_dim, activation="relu")
        self.residual_dense = layers.Dense(hidden_dim, activation="relu")
        self.dropout = layers.Dropout(0.3)
        self.classifier = layers.Dense(num_classes, activation="softmax")

    def call(self, inputs, training=False):
        x = self.dense_in(inputs)
        # Skip connection: F(x) + x
        res = self.residual_dense(x)
        x = layers.add([x, res])
        if training:
            x = self.dropout(x, training=training)
        return self.classifier(x)

# Instantiate and build subclassed model with sample tensor
model_subclass = DynamicResidualMLP(hidden_dim=32, num_classes=3)
sample_in = tf.random.normal([4, 16])
sample_out = model_subclass(sample_in)
print(f"Subclassed Model Output Shape: {sample_out.shape}")

Subclassed Model Output Shape: (4, 3)


### 4. Complex Application 1: Deep Residual Block Architecture via Functional API
Residual connections solve the vanishing gradient problem in deep networks by providing a direct identity highway for gradient backpropagation:
$$\mathbf{y} = \sigma(\text{BatchNorm}(\mathbf{W}_2 \cdot \sigma(\text{BatchNorm}(\mathbf{W}_1 \cdot \mathbf{x} + \mathbf{b}_1)) + \mathbf{b}_2) + \mathbf{x})$$
Below, we construct a modular deep residual network function and verify forward and backward passes.

In [7]:
def residual_block(x, units, dropout_rate=0.2):
    # Standard Pre-Activation Residual Block with skip connection.
    shortcut = x
    # If dimensions do not match, project shortcut to target dimensionality
    if x.shape[-1] != units:
        shortcut = layers.Dense(units, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # First conv/dense step
    h = layers.Dense(units)(x)
    h = layers.BatchNormalization()(h)
    h = layers.Activation("gelu")(h)
    h = layers.Dropout(dropout_rate)(h)

    # Second step
    h = layers.Dense(units)(h)
    h = layers.BatchNormalization()(h)

    # Add identity skip connection
    out = layers.add([shortcut, h])
    return layers.Activation("gelu")(out)

# Build deep 6-block residual architecture
inputs = layers.Input(shape=(20,))
z = layers.Dense(64)(inputs)

# Stack 3 successive residual blocks
for _ in range(3):
    z = residual_block(z, units=64, dropout_rate=0.2)

# Final classification projection
outputs = layers.Dense(5, activation="softmax")(z)

resnet_mlp = models.Model(inputs=inputs, outputs=outputs, name="Deep_ResNet_MLP")
resnet_mlp.compile(
    optimizer=optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=losses.CategoricalCrossentropy(),
    metrics=[metrics.CategoricalAccuracy(name="accuracy")]
)

resnet_mlp.summary()

Model: "Deep_ResNet_MLP"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      1,344 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      4,160 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_7[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      4,160 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_8[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 64)        │          0 │ dense_6[0][0],    │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64)        │          0 │ add_2[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │      4,160 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64)        │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │      4,160 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_10[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 64)        │          0 │ activation_1[0][… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64)        │          0 │ add_3[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 64)        │      4,160 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 28,165 (110.02 KB)

 Trainable params: 27,397 (107.02 KB)

 Non-trainable params: 768 (3.00 KB)

### 5. Complex Application 2: Custom Layer Subclassing with Learnable Scaling
Creating custom layers requires subclassing `keras.layers.Layer`:
1. `__init__()`: Accepts hyperparameters.
2. `build(input_shape)`: Allocates trainable weights (`self.add_weight`) dynamically based on input dimensionality.
3. `call(inputs)`: Implements the mathematical forward-pass transformation.
Below, we implement a **Learnable Scaled Dot-Product Feature Projector** with trainable per-feature scale parameters $\mathbf{\gamma}$.

In [8]:
class LearnableFeatureScaling(layers.Layer):
    # Custom Keras Layer with learnable scale (gamma) and shift (beta) parameters.
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        input_dim = int(input_shape[-1])
        # Trainable projection matrix W
        self.kernel = self.add_weight(
            shape=(input_dim, self.units),
            initializer="glorot_uniform",
            trainable=True,
            name="projection_kernel"
        )
        # Trainable scale multiplier gamma
        self.gamma = self.add_weight(
            shape=(self.units,),
            initializer="ones",
            trainable=True,
            name="gamma_scale"
        )
        # Trainable bias shift beta
        self.beta = self.add_weight(
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
            name="beta_shift"
        )
        super().build(input_shape)

    def call(self, inputs):
        # Linear projection
        projected = tf.matmul(inputs, self.kernel)
        # Element-wise scaling and shifting
        return self.gamma * tf.nn.tanh(projected) + self.beta

# Test custom layer inside a functional model
test_inputs = layers.Input(shape=(10,))
scaled_features = LearnableFeatureScaling(units=16)(test_inputs)
custom_model = models.Model(inputs=test_inputs, outputs=scaled_features)

sample_batch = tf.random.normal([8, 10])
output_batch = custom_model(sample_batch)
print(f"Custom Layer Output Shape: {output_batch.shape}")
print(f"Allocated Trainable Weights: {[w.name for w in custom_model.trainable_weights]}")

Custom Layer Output Shape: (8, 16)
Allocated Trainable Weights: ['projection_kernel', 'gamma_scale', 'beta_shift']
